# Packages & Data

In [ ]:
suppressPackageStartupMessages({
    library(dplyr)
    library(tidyr)
    library(tibble)
    library(ggplot2)
    library(performance)
    library(lme4)
    library(tidyverse)
    library(lmerTest)
    library(broom.mixed)
    library(ggeffects)
    library(emmeans)
    library(patchwork)
    library(cowplot)
    library(modelsummary)
    library(datawizard)
    library(see)
})

In [ ]:
simulate_lmm_data <- function() {
 
set.seed(123)
 
n_subjects <- 30
n_time <- 6
 
subject_df <- tibble(
  subject = factor(1:n_subjects),
  b0 = rnorm(n_subjects, mean = 0, sd = 6),
  b1 = rnorm(n_subjects, mean = 0, sd = 1.2)
)
 
df <- expand_grid(
  subject = factor(1:n_subjects),
  time = 0:(n_time - 1)
) |>
  left_join(subject_df, by = "subject") |>
  mutate(
    y = 20 + 2.5 * time + b0 + b1 * time + rnorm(n(), mean = 0, sd = 2)
  )
  return(df)
  }

## Soma Scan

In [ ]:
# Reading SOMA Scan File
files <- system( "tar -tzf /sharedscratch/maff1-group/data/ADNI_Protein_SOMAscan7k.tar.gz", intern = TRUE )
files

In [ ]:
cmd <- "tar -xOzf /sharedscratch/maff1-group/data/ADNI_Protein_SOMAscan7k.tar.gz CruchagaLab_CSF_SOMAscan7k_Protein_matrix_postQC_20230620.csv"
somascan1 <- read.csv( pipe(cmd) )

In [ ]:
cmd <- "tar -xOzf /sharedscratch/maff1-group/data/ADNI_Protein_SOMAscan7k.tar.gz ADNI_Cruchaga_lab_CSF_SOMAscan7k_analyte_information_20_06_2023.csv"
somascan2 <- read.csv( pipe(cmd) )

In [ ]:
table(somascan2$Target)
# GSK-3 beta x2
# GSK-3 alpha 
# Insulin

In [ ]:
# Finding Codes for GSK-3 beta, GSK-3 alpha, & Insulin
somascan_GSKB <- somascan2 |>
    subset(Target == "GSK-3 beta")
somascan_GSKA <- somascan2 |>
    subset(Target == "GSK-3 alpha")
somascan_INS <- somascan2 |>
    subset(Target == "Insulin")
somascan_GSKB
somascan_GSKA
somascan_INS

In [ ]:
# Finding GSK w/ Somascan 1
CNS <- somascan1[, colnames(somascan1) %in% c('RID','EXAMDATE','VISCODE2', 'X24050.26','X3441.64', 'X4883.56')]
CNS <- CNS |>
    rename(GSK3A = X3441.64,
           GSK3B = X24050.26,
           Insulin = X4883.56)

In [ ]:
# Checking for NA values
nrow(CNS)
cat("GSK3B")
sum(is.na(CNS$GSK3B))

cat("GSK3A")
sum(is.na(CNS$GSK3A))

cat("Insulin")
sum(is.na(CNS$Insulin))

In [ ]:
# Removing NA's
CNS1 <- CNS |>
  drop_na(GSK3B, GSK3A, Insulin)
# Taking only baseline measurements
CNS2 <- CNS1 |>
    subset(VISCODE2 == "bl")
nrow(CNS2)

## Outlier Detection

In [ ]:
# Scaling Variables
CNS2$GSK3A_z <- scale(CNS2$GSK3A)
CNS2$GSK3B_z <- scale(CNS2$GSK3B)
CNS2$INS_z <- scale(CNS2$Insulin)

In [ ]:
vars_to_check <- c("GSK3A_z", "GSK3B_z", "INS_z")

In [ ]:
outlier_result <- check_outliers(
  CNS2[vars_to_check],
  method = "zscore_robust")

In [ ]:
outlier_df <- as.data.frame(outlier_result)
head(outlier_df)

In [ ]:
# Counting the number of outliers
CNS_flagged <- CNS2 |>
  mutate(
    row_id = seq_len(n()),
    outlier_flag_raw = outlier_df$Outlier,
    is_outlier = as.logical(outlier_flag_raw)
  )

CNS_flagged |>
  count(is_outlier)

In [ ]:
# Inspecting Candidiate Outliers
candidate_outliers <- CNS_flagged |>
  filter(is_outlier) |>
  select(row_id, RID, all_of(vars_to_check)) |>
  arrange(RID)

candidate_outliers

In [ ]:
# Capping Function
cap_percentile <- function(x, probs = c(0.01, 0.99), na.rm = TRUE) {
  stopifnot(is.numeric(x), length(probs) == 2)
  qs <- quantile(x, probs = probs, na.rm = na.rm, names = FALSE)
  pmin(pmax(x, qs[1]), qs[2])
}

In [ ]:
CNS3 <- CNS_flagged |>
  mutate(
    GSK3A_capped = cap_percentile(GSK3A_z, probs = c(0.01, 0.99)),
    GSK3B_capped = cap_percentile(GSK3B_z, probs = c(0.01, 0.99)),
    Insulin_capped = cap_percentile(INS_z, probs = c(0.01, 0.99)))

In [ ]:
comparison_tab <- tibble(
  variable = c("GSK3A_z", "GSK3A_capped", "GSK3B_z", "GSKB3_capped", "INS_z", "Insulin_capped"),
  min = c(
    min(CNS3$GSK3A_z, na.rm = TRUE),
    min(CNS3$GSK3A_capped, na.rm = TRUE),
    min(CNS3$GSK3B_z, na.rm = TRUE),
    min(CNS3$GSK3B_capped, na.rm = TRUE),
    min(CNS3$Insulin, na.rm = TRUE),
    min(CNS3$Insulin_capped, na.rm = TRUE)
  ),
  p01 = c(
    quantile(CNS3$GSK3A_z, 0.01, na.rm = TRUE),
    quantile(CNS3$GSK3A_capped, 0.01, na.rm = TRUE),
    quantile(CNS3$GSK3B_z, 0.01, na.rm = TRUE),
    quantile(CNS3$GSK3B_capped, 0.01, na.rm = TRUE),
    quantile(CNS3$Insulin, 0.01, na.rm = TRUE),
    quantile(CNS3$Insulin_capped, 0.01, na.rm = TRUE)
  ),
  median = c(
    median(CNS3$GSK3A_z, na.rm = TRUE),
    median(CNS3$GSK3A_capped, na.rm = TRUE),
    median(CNS3$GSK3B_z, na.rm = TRUE),
    median(CNS3$GSK3B_capped, na.rm = TRUE),
    median(CNS3$Insulin, na.rm = TRUE),
    median(CNS3$Insulin_capped, na.rm = TRUE)
  ),
  p99 = c(
    quantile(CNS3$GSK3A_z, 0.99, na.rm = TRUE),
    quantile(CNS3$GSK3A_capped, 0.99, na.rm = TRUE),
    quantile(CNS3$GSK3B_z, 0.99, na.rm = TRUE),
    quantile(CNS3$GSK3B_capped, 0.99, na.rm = TRUE),
    quantile(CNS3$Insulin, 0.99, na.rm = TRUE),
    quantile(CNS3$Insulin_capped, 0.99, na.rm = TRUE)
  ),
  max = c(
    max(CNS3$GSK3A_z, na.rm = TRUE),
    max(CNS3$GSK3A_capped, na.rm = TRUE),
    max(CNS3$GSK3B_z, na.rm = TRUE),
    max(CNS3$GSK3B_capped, na.rm = TRUE),
    max(CNS3$Insulin, na.rm = TRUE),
    max(CNS3$Insulin_capped, na.rm = TRUE)
  )
)

comparison_tab

In [ ]:
# Reading in Cohort
tbl1 <- read.csv("050_ADNI_Merge_Cohort_V3")

In [ ]:
# Merging GSK2 w/ ADNI Merge Cohort
tbl2 <- merge(tbl1, CNS3, by = "RID")
tbl2 <- tbl2 |>
    rename(CNS_DATE = EXAMDATE.y,
           EXAMDATE = EXAMDATE.x,
           CNS_VISCODE = VISCODE2)
nrow(tbl2)

In [ ]:
# Got error code below in initial analysis so limiting it to only participants w/ 2+ data points
    #Error: number of observations (=843) <= number of random effects (=878) for term (VISCODEnum_z | RID); the random-effects parameters and the residual variance (or scale parameter) are probably unidentifiable

In [ ]:
aggregate1 <- aggregate(VISCODE ~ RID, tbl2, length)
nrow(aggregate1) # 501 Participants w/ CNS biomarker data
aggregate2 <- aggregate1 |>
    subset(VISCODE >= 2)
nrow(aggregate2) # 296 w/ 2+ viscodes
sum(aggregate2$VISCODE)

# Keeping only participants with multiple datapoints
tbl3 <- merge(tbl2, aggregate2, by = "RID")

# Modelling

In [ ]:
# GSK3A Tertiles
tbl3 <- tbl3 |>
    # Dividing GSK A index into tertiles
    transform(GSK3A_tertiles = cut(GSK3A, breaks = quantile(GSK3A, probs = (0:3)/3, na.rm = TRUE), 
                                    labels = c("T1", "T2", "T3"),
                                    include.lowest = TRUE)) 
# GSK3B Tertiles
tbl3 <- tbl3 |>
    # Dividing GSK B index into tertiles
    transform(GSK3B_tertiles = cut(GSK3B, breaks = quantile(GSK3B, probs = (0:3)/3, na.rm = TRUE), 
                                    labels = c("T1", "T2", "T3"),
                                    include.lowest = TRUE)) 
# Insulin Tertiles
tbl3 <- tbl3 |>
    # Dividing GSK B index into tertiles
    transform(Insulin_tertiles = cut(Insulin, breaks = quantile(Insulin, probs = (0:3)/3, na.rm = TRUE), 
                                    labels = c("T1", "T2", "T3"),
                                    include.lowest = TRUE)) 

## Unadjusted

### ABETA

In [ ]:
# GSK3A_capped
GSK3A_A_Unadjusted <- lmer(ABETA_capped ~ VISCODEnum_z * GSK3A_capped + (VISCODEnum_z | RID), data = tbl3)
summary(GSK3A_A_Unadjusted)

In [ ]:
# GSK3B_capped
GSK3B_A_Unadjusted <- lmer(ABETA_capped ~ VISCODEnum_z * GSK3B_capped + (VISCODEnum_z | RID), data = tbl3)
summary(GSK3B_A_Unadjusted)

In [ ]:
# Insulin_capped
Insulin_A_Unadjusted <- lmer(ABETA_capped ~ VISCODEnum_z * Insulin_capped + (VISCODEnum_z | RID), data = tbl3)
summary(Insulin_A_Unadjusted)

## Adjusted

## GSK3A

### ABETA

In [ ]:
# GSK3A_capped
GSK3A_A <- lmer(ABETA_capped ~ VISCODEnum_z * GSK3A_capped + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(GSK3A_A)

In [ ]:
# GSK3A_tertiles
GSK3Atertiles_A <- lmer(ABETA_capped ~ VISCODEnum_z * GSK3A_tertiles + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(GSK3Atertiles_A)

### TAU

In [ ]:
# GSK3A_capped
GSK3A_T <- lmer(TAU_capped ~ VISCODEnum_z * GSK3A_capped + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(GSK3A_T)

In [ ]:
# GSK3A_tertiles
GSK3Atertiles_T <- lmer(TAU_capped ~ VISCODEnum_z * GSK3A_tertiles + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(GSK3Atertiles_T)

### PTAU

In [ ]:
# GSK3A_capped
GSK3A_P <- lmer(PTAU_capped ~ VISCODEnum_z * GSK3A_capped + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(GSK3A_P)

In [ ]:
# GSK3A_tertiles
GSK3Atertiles_P <- lmer(PTAU_capped ~ VISCODEnum_z * GSK3A_tertiles + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(GSK3Atertiles_P)

In [ ]:
confint(GSK3A_A, "GSK3A_capped", level = 0.95)
confint(GSK3A_T, "GSK3A_capped", level = 0.95)
confint(GSK3A_P, "GSK3A_capped", level = 0.95)

## GSK3B

### ABETA

In [ ]:
# GSK3B_capped
GSK3B_A <- lmer(ABETA_capped ~ VISCODEnum_z * GSK3B_capped + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(GSK3B_A)

In [ ]:
# GSK3B_tertiles
GSK3Btertiles_A <- lmer(ABETA_capped ~ VISCODEnum_z * GSK3B_tertiles + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(GSK3Btertiles_A)

### TAU

In [ ]:
# GSK3B_capped
GSK3B_T <- lmer(TAU_capped ~ VISCODEnum_z * GSK3B_capped + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(GSK3B_T)

In [ ]:
# GSK3B_tertiles
GSK3Btertiles_T <- lmer(TAU_capped ~ VISCODEnum_z * GSK3B_tertiles + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(GSK3Btertiles_T)

### PTAU

In [ ]:
# GSK3B_capped
GSK3B_P <- lmer(PTAU_capped ~ VISCODEnum_z * GSK3B_capped + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(GSK3B_P)

In [ ]:
# GSK3B_tertiles
GSK3Btertiles_P <- lmer(PTAU_capped ~ VISCODEnum_z * GSK3B_tertiles + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(GSK3Btertiles_P)

In [ ]:
confint(GSK3B_A, "GSK3B_capped", level = 0.95)
confint(GSK3B_T, "GSK3B_capped", level = 0.95)
confint(GSK3B_P, "GSK3B_capped", level = 0.95)

## Insulin

### ABETA

In [ ]:
# Insulin_capped
Insulin_A <- lmer(ABETA_capped ~ VISCODEnum_z * Insulin_capped + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(Insulin_A)

In [ ]:
# Insulin_tertiles
InsulinTertiles_A <- lmer(ABETA_capped ~ VISCODEnum_z * Insulin_tertiles + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(InsulinTertiles_A)

### TAU

In [ ]:
# Insulin_capped
Insulin_T <- lmer(TAU_capped ~ VISCODEnum_z * Insulin_capped + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(Insulin_T)

In [ ]:
# Insulin_tertiles
InsulinTertiles_T <- lmer(TAU_capped ~ VISCODEnum_z * Insulin_tertiles + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(InsulinTertiles_T)

### PTAU

In [ ]:
# Insulin_capped
Insulin_P <- lmer(PTAU_capped ~ VISCODEnum_z * Insulin_capped + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(Insulin_P)

In [ ]:
# Insulin_tertiles
InsulinTertiles_P <- lmer(PTAU_capped ~ VISCODEnum_z * Insulin_tertiles + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(InsulinTertiles_P)

In [ ]:
confint(Insulin_A, "Insulin_capped", level = 0.95)
confint(Insulin_T, "Insulin_capped", level = 0.95)
confint(Insulin_P, "Insulin_capped", level = 0.95)

## METS-IR

In [ ]:
tbl1 <- tbl1 |>
    # Dividing METS-IR into tertiles
    transform(METSIR_tertiles = cut(METS_IR, breaks = quantile(METS_IR, probs = (0:3)/3, na.rm = TRUE), 
                                    labels = c("T1", "T2", "T3"),
                                    include.lowest = TRUE)) 

### ABETA

In [ ]:
# METSIR_capped
METSIR_A <- lmer(ABETA_capped ~ VISCODEnum_z * METSIR_capped + AGE_z + PTGENDER + DIAGNOSIS_bl + APOE4 + PTEDUCAT_z + BMI_capped +(VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))

In [ ]:
# METSIR_tertiles
METSIRtertiles_A <- lmer(ABETA_capped ~ VISCODEnum_z * METSIR_tertiles + AGE_z + PTGENDER + DIAGNOSIS_bl + APOE4 + PTEDUCAT_z + BMI_capped +(VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))

### TAU

In [ ]:
# METSIR_capped
METSIR_T <- lmer(TAU_capped ~ VISCODEnum_z * METSIR_capped + AGE_z + PTGENDER + DIAGNOSIS_bl + APOE4 + PTEDUCAT_z + BMI_capped +(VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))

### PTAU

In [ ]:
# METSIR_capped
METSIR_P <- lmer(PTAU_capped ~ VISCODEnum_z * METSIR_capped + AGE_z + PTGENDER + DIAGNOSIS_bl + APOE4 + PTEDUCAT_z + BMI_capped +(VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))

## Model Comparison

### Comparing AB Models

In [ ]:
# Including METS-IR (best peripheral biomarker), GSKA, GSKB, and Insulin

In [ ]:
# Scaled values
modelsummary(
  list(
    "Adjusted METS-IR Model" = METSIR_A,
    "Adjusted GSK3A Model" = GSK3A_A,
    "Adjusted GSK3B Model" = GSK3B_A,
    "Adjusted Insulin Model" = Insulin_A
  ),
  statistic = "({std.error})"
)

### Comparing TAU Models

In [ ]:
# Scaled values
modelsummary(
  list(
    "Adjusted METS-IR Model" = METSIR_T,
    "Adjusted GSK3A Model" = GSK3A_T,
    "Adjusted GSK3B Model" = GSK3B_T,
    "Adjusted Insulin Model" = Insulin_T
  ),
  statistic = "({std.error})"
)

In [ ]:
# Scaled values
modelsummary(
  list(
    "Adjusted METS-IR Model" = METSIR_P,
    "Adjusted GSK3A Model" = GSK3A_P,
    "Adjusted GSK3B Model" = GSK3B_P,
    "Adjusted Insulin Model" = Insulin_P
  ),
  statistic = "({std.error})"
)

In [ ]:
# Tertiles
modelsummary(
  list(
    "Adjusted METS-IR Model" = METSIRtertiles_A,
    "Adjusted GSK3A Model" = GSK3Atertiles_A,
    "Adjusted GSK3B Model" = GSK3Btertiles_A,
    "Adjusted Insulin Model" = InsulinTertiles_A
  ),
  statistic = "({std.error})"
)

# Graphing

In [ ]:
# GSK-3A
g1 <- ggpredict(GSK3Atertiles_A, terms = c("VISCODEnum_z", "GSK3A_tertiles")) |>
    plot() +
    labs(
        title = "Predicted AB Over Time Stratified by GSK-3A Tertile",
        x = "Months Since Baseline (scaled)",
        y = "Predicted AB (pg/mL)",
        color = "CSF GSK-3A"
      ) +
  theme_minimal(base_size = 13)
g1
# Shows that higher GSKA associated with lower CSF AB

In [ ]:
# GSK-3B
g2 <- ggpredict(GSK3Btertiles_A, terms = c("VISCODEnum_z", "GSK3B_tertiles")) |>
    plot() +
    labs(
        title = "Predicted AB Over Time Stratified by GSK-3B Tertile",
        x = "Months Since Baseline (scaled)",
        y = "Predicted AB (pg/mL)",
        color = "CSF GSK-3B"
      ) +
  theme_minimal(base_size = 13)
g2
# Shows that higher GSKB associated with lower CSF AB

In [ ]:
# Insulin
g3 <- ggpredict(InsulinTertiles_A, terms = c("VISCODEnum_z", "Insulin_tertiles")) |>
    plot() +
    labs(
        title = "Predicted AB Over Time Stratified by Insulin Tertile",
        x = "Months Since Baseline (scaled)",
        y = "Predicted AB (pg/mL)",
        color = "CSF Insulin"
      ) +
  theme_minimal(base_size = 13)
g3
# Shows that higher Insulin associated with higher CSF AB

In [ ]:
ggsave("059_AB x Time x GSK-3A Tertile.png", g1, width = 10, height = 5)
ggsave("060_AB x Time x GSK-3B Tertile.png", g2, width = 10, height = 5)
ggsave("061_AB x Time x Insulin Tertile.png", g3, width = 10, height = 5)

In [ ]:
write.csv(tbl3, "063_Phase_II_Cohort")